# Avoid Hands — Colab launcher con auto-resume

Questo notebook usa la versione del progetto che:
- carica `dqn_latest.weights.h5` **prima del training**, se esiste;
- usa `dqn_best.keras` come fallback;
- per la prediction/video usa per default il **latest**, con fallback sul best.


In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Estrai/aggiorna il progetto

Carica `avoid_hands-dqn-auto-resume.zip` direttamente in `MyDrive`.

Questa cella aggiorna i file di codice ma **non cancella** la cartella `checkpoints/` già presente, quindi i modelli salvati restano disponibili per il resume.


In [ ]:
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/drive/MyDrive/avoid_hands-dqn-auto-resume.zip")
PROJECT_ROOT = Path("/content/drive/MyDrive/avoid_hands_project/avoid_hands-master")

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"ZIP non trovato: {ZIP_PATH}")

PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(PROJECT_ROOT.parent)

print("Codice aggiornato. I checkpoint esistenti non sono stati cancellati.")
print(PROJECT_ROOT)


In [ ]:
%cd /content/drive/MyDrive/avoid_hands_project/avoid_hands-master

import os
from pathlib import Path

os.environ["PYTHONPATH"] = str(Path.cwd() / "dont_touch_my_presents")
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["XDG_RUNTIME_DIR"] = "/tmp/runtime-root"
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"

Path("/tmp/runtime-root").mkdir(parents=True, exist_ok=True)


In [ ]:
!pip install -q -r requirements.txt


## Controlla cosa verrà caricato

Il training userà automaticamente:
1. `checkpoints/dqn_latest.weights.h5`, se presente;
2. altrimenti `checkpoints/dqn_best.keras`;
3. altrimenti partirà da zero.


In [ ]:
from pathlib import Path

latest = Path("checkpoints/dqn_latest.weights.h5")
best = Path("checkpoints/dqn_best.keras")

print("latest:", latest.exists(), latest)
print("best:  ", best.exists(), best)


## Training / retraining

Il caricamento del checkpoint avviene dentro `train_dqn.py` **prima del primo step di training**.


In [ ]:
!python -u train_dqn.py


## Prediction + video

Per default usa il checkpoint **latest**; se manca, usa il **best**.


In [ ]:
!python -u record_best_run.py --algorithm dqn --episodes 1


Per forzare il best model invece del latest:


In [ ]:
!python -u record_best_run.py --algorithm dqn --checkpoint best --episodes 1
